# PCA on median flux profiles

Eight primary models; the union model is omitted. Each `medianFlux` value is the
mean of four within-chain medians. The shared reaction core is derived from the
input profiles after the 93 fixed exclusions (303 reactions in the recorded data).
Reactions are centred and scaled across the eight models before seven-component
PCA using deterministic full SVD. Relative paths resolve from the repository root.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

INPUT_DIR = Path(".")
OUTPUT_DIR = Path("results")
PCA_DIR = INPUT_DIR / "CHRR_sampling" / "pca"
STAGES = ["egg_c", "L1_c", "L2_c", "L3_c", "L4f_c", "L4m_c", "Am_c", "Af_c"]

# Fixed reaction exclusions recorded for the sampling analysis.
EXCLUDED_REACTIONS = """
MAR00023 MAR01506 MAR01507 MAR02960 MAR02993 MAR03031 MAR03036 MAR03041
MAR03131 MAR06630 MAR06631 MAR06632 MAR06633 MAR06634 MAR06635 MAR06636
MAR06637 MAR06638 MAR06639 MAR06641 MAR06642 MAR06646 MAR06647 MAR06648
MAR06649 MAR06651 MAR06652 MAR06653 MAR06654 MAR06655 MAR06656 MAR06657
MAR06658 MAR06659 MAR06660 MAR06662 MAR06663 MAR06664 MAR06665 MAR06666
MAR06667 MAR06668 MAR06669 MAR06670 MAR06671 MAR06672 MAR06673 MAR06674
MAR06675 MAR06676 MAR06677 MAR06678 MAR06680 MAR06682 MAR06685 MAR06686
MAR06687 MAR06690 MAR06691 MAR06692 MAR06693 MAR06694 MAR06695 MAR06697
MAR06699 MAR06700 MAR06701 MAR06702 MAR06703 MAR06704 MAR06705 MAR08697
MAR08698 MAR08699 MAR08702 MAR08703 MAR08704 MAR08706 MAR08709 MAR08710
MAR08711 MAR08712 MAR08716 MAR11894 MAR11895 MAR11896 MAR11897 MAR20001
MAR20003 MAR20005 MAR20006 MAR20007 MAR20008
""".split()


In [ ]:
# The exported profiles already contain the means of four chain medians.
profiles = [
    pd.read_csv(PCA_DIR / f"{stage}_median.csv").set_index("rxnID")["medianFlux"].rename(stage)
    for stage in STAGES
]
matrix = pd.concat(profiles, axis=1, join="inner")
matrix = matrix.drop(index=EXCLUDED_REACTIONS, errors="ignore")


In [ ]:
# Annotations are optional and do not determine reaction eligibility.
annotations = pd.DataFrame(index=pd.Index([], name="rxnID"))
annotation_path = PCA_DIR / "reaction_evidence_table.csv"
if annotation_path.exists():
    annotations = pd.read_csv(annotation_path).drop_duplicates("rxnID").set_index("rxnID")
    columns = [c for c in ["probable_name", "subSystem", "eccodes", "equation"] if c in annotations]
    annotations = annotations[columns].rename(columns={"probable_name": "rxnName"})


In [ ]:
# Models are observations; each reaction is standardized across eight models.
scaled = StandardScaler().fit_transform(matrix.T.values)


In [ ]:
pca = PCA(n_components=len(STAGES) - 1, svd_solver="full")
pc_names = [f"PC{i+1}" for i in range(len(STAGES) - 1)]
scores = pd.DataFrame(pca.fit_transform(scaled), index=STAGES, columns=pc_names)
loadings = pd.DataFrame(pca.components_.T, index=matrix.index, columns=pc_names)
loadings = loadings.join(annotations)
variance = pd.DataFrame({
    "component": pc_names,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "explained_variance_percent": 100 * pca.explained_variance_ratio_,
    "cumulative_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
})


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
scores.rename_axis("model").to_csv(OUTPUT_DIR / "pca_scores.csv", float_format="%.17g")
loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv", index_label="rxnID", float_format="%.17g")
variance.to_csv(OUTPUT_DIR / "pca_explained_variance.csv", index=False, float_format="%.17g")
